# E-commerce KPI Calculations and Validation

**Goal:** calculate core sales, profitability, customer, operations, and marketing KPIs from `sample_ecommerce_data.csv`.

The dataset is synthetic. KPI definitions and assumptions are documented in `kpi_dictionary.md`. Run this notebook from the project folder so the relative CSV path works.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("sample_ecommerce_data.csv")
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
display(df.head())
display(df.info())

Rows: 120


,order_id,order_date,customer_id,product_category,quantity,unit_price,unit_cost,discount_pct,shipping_fee,order_status,is_returned,marketing_channel,marketing_cost
0,ORD1001,2026-03-18,CUST002,Clothing,1,58.49,44.45,0.15,9.99,Completed,False,Paid Search,17.91
1,ORD1002,2026-03-09,CUST046,Electronics,2,287.01,157.80,0.10,9.99,Completed,False,Social Ads,19.49
2,ORD1003,2026-01-25,CUST018,Clothing,1,45.22,30.55,0.15,4.99,Completed,False,Paid Search,7.71
3,ORD1004,2026-01-18,CUST062,Beauty,4,14.00,7.00,0.10,9.99,Completed,False,Organic,0.00
4,ORD1005,2026-02-11,CUST009,Clothing,4,83.84,44.41,0.00,0.00,Completed,True,Organic,0.00


<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           120 non-null    str    
 1   order_date         120 non-null    str    
 2   customer_id        120 non-null    str    
 3   product_category   120 non-null    str    
 4   quantity           120 non-null    int64  
 5   unit_price         120 non-null    float64
 6   unit_cost          120 non-null    float64
 7   discount_pct       120 non-null    float64
 8   shipping_fee       120 non-null    float64
 9   order_status       120 non-null    str    
 10  is_returned        120 non-null    bool   
 11  marketing_channel  120 non-null    str    
 12  marketing_cost     120 non-null    float64
dtypes: bool(1), float64(5), int64(1), str(6)
memory usage: 11.5 KB


None

## 1. Data quality checks

In [2]:
# Normalize data types and validate the key fields.
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["is_returned"] = df["is_returned"].astype(bool)

checks = {
    "duplicate_order_ids": int(df["order_id"].duplicated().sum()),
    "missing_order_ids": int(df["order_id"].isna().sum()),
    "invalid_dates": int(df["order_date"].isna().sum()),
    "nonpositive_quantity": int((df["quantity"] <= 0).sum()),
    "negative_unit_price": int((df["unit_price"] < 0).sum()),
    "negative_unit_cost": int((df["unit_cost"] < 0).sum()),
    "discount_out_of_range": int((~df["discount_pct"].between(0, 1)).sum()),
    "invalid_status": int((~df["order_status"].isin(["Completed", "Cancelled"])).sum()),
    "cancelled_marked_returned": int(((df["order_status"] == "Cancelled") & df["is_returned"]).sum()),
}
pd.Series(checks, name="count").to_frame()

,count
duplicate_order_ids,0
missing_order_ids,0
invalid_dates,0
nonpositive_quantity,0
negative_unit_price,0
negative_unit_cost,0
discount_out_of_range,0
invalid_status,0
cancelled_marked_returned,0


In [3]:
# Derived fields. Discounts are applied to item value; shipping is added afterward.
df["item_gmv"] = df["quantity"] * df["unit_price"]
df["discount_amount"] = df["item_gmv"] * df["discount_pct"]
df["item_revenue_after_discount"] = df["item_gmv"] - df["discount_amount"]
df["order_revenue"] = df["item_revenue_after_discount"] + df["shipping_fee"]
df["estimated_cogs"] = df["quantity"] * df["unit_cost"]

completed = df.loc[df["order_status"].eq("Completed")].copy()
retained = completed.loc[~completed["is_returned"]].copy()

print(f"All orders: {len(df)}")
print(f"Completed orders: {len(completed)}")
print(f"Cancelled orders: {(df['order_status'] == 'Cancelled').sum()}")
print(f"Returned completed orders: {completed['is_returned'].sum()}")
print(f"Retained completed orders: {len(retained)}")

All orders: 120
Completed orders: 111
Cancelled orders: 9
Returned completed orders: 7
Retained completed orders: 104


## 2. Core KPI calculations

In [4]:
def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else np.nan

gmv = completed["item_gmv"].sum()
net_sales = retained["order_revenue"].sum()
retained_orders = retained["order_id"].nunique()
completed_orders = completed["order_id"].nunique()
all_orders = df["order_id"].nunique()
retained_customers = retained["customer_id"].nunique()
repeat_customers = int((retained.groupby("customer_id")["order_id"].nunique() >= 2).sum())
cogs = retained["estimated_cogs"].sum()
gross_profit = net_sales - cogs
marketing_cost = retained["marketing_cost"].sum()

kpis = pd.DataFrame({
    "KPI": [
        "GMV before discounts",
        "Net sales revenue",
        "Completed, non-returned orders",
        "Average order value (AOV)",
        "Units sold (retained orders)",
        "Estimated COGS",
        "Estimated gross profit",
        "Estimated gross margin (%)",
        "Cancellation rate (%)",
        "Return rate (%)",
        "Repeat-customer rate (%)",
        "Average discount (%)",
        "Marketing cost per retained order",
        "Estimated contribution after product and attributed marketing cost",
        "ROAS-like ratio (net sales / attributed marketing cost)",
    ],
    "Value": [
        gmv,
        net_sales,
        retained_orders,
        safe_divide(net_sales, retained_orders),
        retained["quantity"].sum(),
        cogs,
        gross_profit,
        safe_divide(gross_profit, net_sales) * 100,
        safe_divide((df["order_status"] == "Cancelled").sum(), all_orders) * 100,
        safe_divide(completed["is_returned"].sum(), completed_orders) * 100,
        safe_divide(repeat_customers, retained_customers) * 100,
        retained["discount_pct"].mean() * 100,
        safe_divide(marketing_cost, retained_orders),
        net_sales - cogs - marketing_cost,
        safe_divide(net_sales, marketing_cost),
    ]
})
kpis

,KPI,Value
0,GMV before discounts,22467.590000
1,Net sales revenue,20156.768000
2,"Completed, non-returned orders",104.000000
3,Average order value (AOV),193.815077
4,Units sold (retained orders),257.000000
5,Estimated COGS,13122.070000
6,Estimated gross profit,7034.698000
7,Estimated gross margin (%),34.899930
8,Cancellation rate (%),7.500000
9,Return rate (%),6.306306


## 3. Segment analysis

In [5]:
# Category-level results, using retained completed orders.
category_kpis = (
    retained.groupby("product_category")
    .agg(
        orders=("order_id", "nunique"),
        units=("quantity", "sum"),
        net_sales=("order_revenue", "sum"),
        estimated_cogs=("estimated_cogs", "sum"),
        avg_discount_pct=("discount_pct", "mean"),
    )
    .reset_index()
)
category_kpis["estimated_gross_profit"] = category_kpis["net_sales"] - category_kpis["estimated_cogs"]
category_kpis["estimated_gross_margin_pct"] = (
    category_kpis["estimated_gross_profit"] / category_kpis["net_sales"] * 100
)
category_kpis.sort_values("net_sales", ascending=False)

,product_category,orders,units,net_sales,estimated_cogs,avg_discount_pct,estimated_gross_profit,estimated_gross_margin_pct
3,Home,22,64,6511.2615,4393.39,0.081818,2117.8715,32.526285
2,Electronics,18,41,6108.7540,3965.48,0.077778,2143.2740,35.085289
4,Sports,19,38,2776.3225,1708.88,0.089474,1067.4425,38.448073
1,Clothing,23,55,2544.4660,1602.89,0.065217,941.5760,37.004857
0,Beauty,22,59,2215.9640,1451.43,0.093182,764.5340,34.501192


In [6]:
# Channel-level directional view. Costs are illustrative order-attributed costs.
channel_kpis = (
    retained.groupby("marketing_channel")
    .agg(
        orders=("order_id", "nunique"),
        net_sales=("order_revenue", "sum"),
        attributed_marketing_cost=("marketing_cost", "sum"),
        estimated_cogs=("estimated_cogs", "sum"),
    )
    .reset_index()
)
channel_kpis["estimated_gross_profit"] = channel_kpis["net_sales"] - channel_kpis["estimated_cogs"]
channel_kpis["marketing_cost_per_order"] = (
    channel_kpis["attributed_marketing_cost"] / channel_kpis["orders"].replace(0, np.nan)
)
channel_kpis["roas_like"] = (
    channel_kpis["net_sales"] / channel_kpis["attributed_marketing_cost"].replace(0, np.nan)
)
channel_kpis.sort_values("net_sales", ascending=False)

,marketing_channel,orders,net_sales,attributed_marketing_cost,estimated_cogs,estimated_gross_profit,marketing_cost_per_order,roas_like
2,Paid Search,33,5860.4130,409.89,3922.16,1938.2530,12.420909,14.297526
1,Organic,29,5060.4095,0.00,3195.42,1864.9895,0.000000,NaN
0,Email,16,3753.5255,0.00,2489.13,1264.3955,0.000000,NaN
4,Social Ads,15,2993.3995,166.48,1949.80,1043.5995,11.098667,17.980535
3,Referral,11,2489.0205,0.00,1565.56,923.4605,0.000000,NaN


## 4. Validation and interpretation prompts

In [7]:
# Basic consistency checks; these should pass for the supplied sample data.
assert df["order_id"].is_unique, "Each row should represent a unique order."
assert df["quantity"].gt(0).all(), "Quantity must be positive."
assert df["discount_pct"].between(0, 1).all(), "Discount must be between 0 and 1."
assert not ((df["order_status"] == "Cancelled") & df["is_returned"]).any(), "Cancelled orders should not be returned."
assert (retained["order_revenue"] >= 0).all(), "Retained order revenue should be non-negative."

print("Validation checks passed.")
print("Interpretation reminders:")
print("- AOV should be reviewed alongside estimated gross margin.")
print("- Cancellation rate uses all orders; return rate uses completed orders.")
print("- ROAS-like ratio is not full channel ROAS because the sample costs are illustrative.")
print("- Conversion rate and CAC cannot be calculated without sessions and complete acquisition spend.")

Validation checks passed.
Interpretation reminders:
- AOV should be reviewed alongside estimated gross margin.
- Cancellation rate uses all orders; return rate uses completed orders.
- ROAS-like ratio is not full channel ROAS because the sample costs are illustrative.
- Conversion rate and CAC cannot be calculated without sessions and complete acquisition spend.


## 5. Business interpretation

Use the generated tables to answer:
1. Which category contributes the most net sales? Does it also have the strongest gross margin?
2. Is the cancellation rate high enough to investigate by category, date, or channel?
3. How does average discount relate to estimated gross margin?
4. Which channels have the highest sales, and are the attributed cost values complete enough for a fair comparison?
5. What additional data would be required before recommending a business action?

Avoid claiming causation from these descriptive metrics. Check data quality, sample size, and business context before making decisions.